In [276]:
import pandas as pd
from vertexai.generative_models import (
    FunctionDeclaration,
    GenerationConfig,
    GenerativeModel,
    Tool,
    HarmCategory,
    HarmBlockThreshold
)
import json
import re
import pickle


In [277]:
materials_df = pd.read_csv('data/relevant_materials.csv')
materials = materials_df.to_json(orient='records')

In [278]:
PROJECT_ID = "proj-sales-recommender-dev"
LOCATION = "us-central1" 

import vertexai

vertexai.init(project=PROJECT_ID, location=LOCATION)


In [279]:
MODEL_ID = "gemini-2.0-flash-001"

model = GenerativeModel(
    MODEL_ID,
    safety_settings={
            HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
            HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
        },

)

In [280]:
with open("data/eligible_for_classification_60.pkl", 'rb') as file:
    eligible_for_classification = pickle.load(file)

In [281]:
def prompt_v1(cc_project_json, materials, product, product_priority):
    question = f'''

    **Objective:** Classify ConstructConnect projects as very high, high, moderate, low, very low, or not relevant.

        **Instructions:**

        1. **Analyze the provided JSON data:** Understand the project details, including relevant fields and data points.  The JSON data representing ConstructConnect projectis provided **Project Data**.

        2. **Identify Key Materials:** Extract the key materials used for each project across multiple columns in the project data. Determine if these materials are present in the provided "Relevant Materials" list.

        3. **Consider Product Priority:**  Use the provided "Product Priority" to weigh the classifying attributes.  Higher priority products should be given more weight in the classification process.

        4. **Classify Projects:**
            * Prioritize projects containing materials relevant to the project AND matching the "Relevant Materials" list.
            * Classify based on the "Product Priority."
            * When other factors are equal, prioritize higher-value projects (e.g., higher total dollar amount).

        5. **Estimate Relevancy:** Estimate the relevancy of each project based on the "Product Priority," materials match, and total dollar amount.

        6. **Respond in JSON:** Return ONLY the JSON response, as shown in the example below.  The JSON should be a list of project evaluations, classifying each project as "very high," "high," "moderate," "low," "very low," or "not relevant."  Include the reasoning behind each classification.

        **Input Data:**

        **Relevant Data:**
        {cc_project_json}
    

        **Relevant Materials:**
        {materials}

        ** Product Priority:**
        {product_priority}

        **Example Output:**
        {{"ProjectID": 1006193703,"Product": {product}, "Product priority":{product_priority}, "Materials matching": [material1, material2, material3],  "Relevance Classification": High, "Reasoning": reason for classification}}

        '''
    prompt = question
    contents = [prompt]

    # Generate text using non-streaming method
    response = model.generate_content(contents)
    return response.text


In [282]:
def prompt_v2(cc_project_json, materials, product, product_priority):
    question = f'''
        <prompt>
        <objective>Classify ConstructConnect projects as very high, high, moderate, low, very low, or not relevant.</objective>
        <instructions>
            <step>
                <description>Analyze the provided JSON data:</description>
                <details>Understand the project details, including relevant fields and data points. The JSON data representing ConstructConnect projects is provided as "Project Data".</details>
            </step>
            <step>
                <description>Identify Key Materials:</description>
                <details>Extract the key materials used for each project across multiple columns in the project data. Determine if these materials are present in the provided "Relevant Materials" list.</details>
            </step>
            <step>
                <description>Consider Product Priority:</description>
                <details>Use the provided "Product Priority" to weigh the classifying attributes. Higher priority products should be given more weight in the classification process.</details>
            </step>
            <step>
                <description>Classify Projects:</description>
                <details>
                    <point>Prioritize projects containing materials relevant to the project AND matching the "Relevant Materials" list.</point>
                    <point>Classify based on the "Product Priority."</point>
                    <point>When other factors are equal, prioritize higher-value projects (e.g., higher total dollar amount).</point>
                </details>
            </step>
            <step>
                <description>Respond in JSON:</description>
                <details>Return ONLY the JSON response, as shown in the example below. The JSON should be a list of project evaluations, classifying each project as "very high," "high," "moderate," "low," "very low," or "not relevant." Include the reasoning behind each classification.</details>
            </step>
        </instructions>
        <input_data>
            <project_data>
                <description>Project Data</description>
                <value>{cc_project_json}</value>
            </project_data>
            <relevant_materials>
                <description>Relevant Materials</description>
                <value>{materials}</value>
            </relevant_materials>
            <product_priority>
                <description>Product Priority</description>
                <value>{product_priority}</value>
            </product_priority>
        </input_data>
        <example_output>
            <json_example>
                {{"ProjectID": 1006193703,"Product": {product}, "Product priority":{product_priority}, "Materials matching": [material1, material2, material3],  "Relevance Classification": High, "Reasoning": reason for classification}}
            </json_example>
        </example_output>
        </prompt>
    
        '''
    prompt = question
    contents = [prompt]

    # Generate text using non-streaming method
    response = model.generate_content(contents)
    return response.text


In [309]:
def prompt_v3(cc_project_json, materials, product, product_priority):

  
  question = f'''
  <prompt>
    <objective>Classify ConstructConnect projects as Very high, high, moderate, low and Not relevant based on material relevance and product priority.</objective>
    <instructions>
      <step>
        <description>Analyze Project Data:</description>
        <details>
          <point>Parse the provided JSON data representing ConstructConnect projects.</point>
          <point>Identify and extract the following fields for each project: ProjectID, Product, relevant material columns (specify the names of these columns), Product Priority and total dollar amount (if available).</point>
        </details>
      </step>
      <step>
        <description>Identify Key Materials:</description>
        <details>
          <point>For each project, extract all material names listed in multiple columns.</point>
          <point>Create a list of unique material names for each project.</point>
          <oint>Out of the unique material names, identify the materials that are most relevant to the project and the construction type of the project.</point>
          <point>Compare each unique relevant material name against the provided "Relevant Materials" list.</point>
          <point>Create a new list containing only the materials from the project that are also present in the "Relevant Materials" list.  This is the "Materials Matching" list.</point>
          <point>If no materials match, the "Materials Matching" list should be empty.</point>
        </details>
      </step>
      <step>
        <description>Consider Product Priority:</description>
        <details>
          <point>For each project, the product priority is provided</point>
          <point>Assign a numerical weight to each priority level (e.g., High = 5, Low = 1, Not Relevant = 0).  This weight will be used in the classification process.</point>
        </details>
      </step>
      <step>
        <description>Classify Projects:</description>
        <details>
          <point>Classify each project as "very high", "high", "moderate", "low" and "Not relevant".</point>
          <point>If the "Materials Matching" list for a project is empty, classify the project as "Not Relevant" and proceed to the next project.</point>
          <point>If the "Materials Matching" list is not empty, proceed with the following steps:</point>
          <point>Analyze the list of "Materials Matching"</point>
          <point>Consider the Product priority and the weight assigned in the previous step</point>
          <point>Higher dollar amounts should result in a higher final classification.</point>
          <point>Assign the final classification to the project.</point>
        </details>
      </step>
      <step>
        <description>Respond in JSON:</description>
        <details>
          <point>Return ONLY a JSON object.</point>
          <point>The JSON object should have the following structure:</point>
              {{"ProjectID": 1006193703,"Product": {product}, "Product priority":{product_priority}, "Materials matching": [material1, material2, material3],  "Relevance Classification": High, "Reasoning": reason for classification}}
          <point>The "Reasoning" field should explain the steps taken to classify the project, including the matched materials, product priority, and any consideration of the total dollar amount.</point>
        </details>
      </step>
    </instructions>
    <input_data>
      <project_data>
        <description>Project Data</description>
        <value>{cc_project_json}</value>
      </project_data>
      <relevant_materials>
        <description>Relevant Materials</description>
        <value>{materials}</value>
      </relevant_materials>
      <product_priority>
        <description>Product Priority</description>
        <value>{product_priority}</value>
      </product_priority>
    </input_data>
    <example_output>
      <json_example>
       {{"ProjectID": {cc_project_json.get('ProjectID')}, "Product": "{product}", "Product priority": "{product_priority}", "Materials matching": [],  "Relevance Classification": "", "Reasoning": ""}}
      </json_example>
    </example_output>
  </prompt>
  '''
  prompt = question
  contents = [prompt]

  # Generate text using non-streaming method
  response = model.generate_content(contents)
  return response.text
  

In [284]:
def prompt_v4(cc_project_json, materials, product, product_priority):
  question = f'''
        You are a construction projects classifying assistant. You are given project data and supporting information to classify the project based on that information.
        Following are the classifications and their descriptions:

        <classifications>
        VERY HIGH: Materials listed in the project data are the *most* important materials for the project and the construction type. 
        These materials are also present in the relevant materials list provided in the <INPUTS>. 
        The product associated with the project is of *high* priority. 
        The valuation of the project is *very high*.

        HIGH: Materials listed in the project data are *important* materials for the project and the construction type. 
        These materials are also present in the relevant materials list provided in the <INPUTS>. 
        The product associated with the project is of *high* priority. 
        The valuation of the project is *high*.

        MODERATE: Materials listed in the project data are *somewhat important* materials for the project and the construction type.  
        These materials are present in the relevant materials list provided in the <INPUTS>. 
        The product associated with the project is of *medium* priority. 
        The valuation of the project is *moderate*.

        LOW: Materials listed in the project data are *less important* materials for the project and the construction type. 
        These materials are present in the relevant materials list provided in the <INPUTS>. 
        The product associated with the project is of *low* priority. 
        The valuation of the project is *low*.

        NOT RELEVANT: Materials listed in the project data are *not important* materials for the project and the construction type. 
        These materials are *not* present in the relevant materials list provided in the <INPUTS>. 
        The product associated with the project is of *low* priority. 
        The valuation of the project is *very low*.
        </classifications>

        Following are the inputs:

        <inputs>
            <project_data>
                <description>Project Data</description>
                <value>{cc_project_json}</value>
            </project_data>
            <relevant_materials>
                <description>Relevant Materials</description>
                <value>{materials}</value>
            </relevant_materials>
            <product_priority>
                <description>Product Priority (High, Low)</description>
                <value>{product_priority}</value>
            </product_priority>
        </inputs>

        <rules>
            * Think through your reasoning before making the classification and place your thought process in <thinking></thinking> tags. 
            This is your space to think and reason about the project classification.  
            Consider the importance of the materials for the specific project, their presence in the relevant materials list, the product priority, and the project valuation to make your decision.
            * Once you have finished thinking, classify the project using ONLY the classifications listed above and place it in <answer></answer> tags.
        </rules>

        Format the output as below
        <output>
            <project_id></project_id>
            <product></product>
            <product_priority></product_priority>
            <materials_matching></materials_matching>
            <classification></classification>
            <thinking></thinking>
        </output>

  '''
  prompt = question
  contents = [prompt]

  # Generate text using non-streaming method
  response = model.generate_content(contents)
  return response.text
  

In [303]:
import pandas as pd
import json

def process_and_save_prompt_output(prompt_function, csv_filename, eligible_for_classification, materials):
   
    product_responses = []
    for index, row in eligible_for_classification.iterrows():
        row_json_str = row.to_json()
        cc_project_json_record = json.loads(row_json_str)

        output = prompt_function(cc_project_json_record, materials, cc_project_json_record['PRODUCT'], cc_project_json_record['PRODUCT_PRIORITY'])
        otpt = output.replace("```json", "").replace("```", "")  # Clean up any code block markers
        product_responses.append(otpt)

    all_data = []
    for item in product_responses:
        try:
            json_list = json.loads(item)
            all_data.extend(json_list)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")
            print(f"Problematic string: {item}")
            continue

    df_single_prompt_output = pd.DataFrame(all_data)
    df_single_prompt_output.to_csv(csv_filename, index=False)

In [294]:
process_and_save_prompt_output(prompt_v1, 'data/single_prompt_variations/bullet_point_promot_output_60.csv', eligible_for_classification, materials)


In [ ]:
process_and_save_prompt_output(prompt_v2, 'data/single_prompt_variations/xml_prompt_output_60.csv', eligible_for_classification, materials)


In [310]:
process_and_save_prompt_output(prompt_v3, 'data/single_prompt_variations/prompt_detailed_steps_output_60.csv', eligible_for_classification, materials)


In [298]:
import pandas as pd
import xml.etree.ElementTree as ET

def add_xml_to_dataframe(xml_string, df):
    """
    Parses XML string, extracts data, and adds it as a row to a DataFrame.

    Args:
        xml_string: The XML string to parse.
        df: The Pandas DataFrame to add the data to.  If the DataFrame is empty,
           it will be initialized with the correct columns.

    Returns:
        The updated Pandas DataFrame.
    """
    try:
        root = ET.fromstring(xml_string)
        data = {}

        # Extract data from XML elements, handling potential missing elements
        data['project_id'] = root.find('project_id').text if root.find('project_id') is not None else None
        data['product'] = root.find('product').text if root.find('product') is not None else None
        data['product_priority'] = root.find('product_priority').text if root.find('product_priority') is not None else None
        data['materials_matching'] = root.find('materials_matching').text if root.find('materials_matching') is not None else None
        data['classification'] = root.find('classification').text if root.find('classification') is not None else None
        data['thinking'] = root.find('thinking').text if root.find('thinking') is not None else None

        # Check if the DataFrame is empty. If so, create it with the correct columns.
        if df.empty:
          df = pd.DataFrame(columns=data.keys())  # Initialize with column names


        df = pd.concat([df, pd.DataFrame([data])], ignore_index=True) #More efficient than append
        return df

    except ET.ParseError as e:
        print(f"Error parsing XML: {e}")
        return df  # Return the DataFrame unchanged in case of error
    except Exception as e: # Catch other potential errors
        print(f"An error occurred: {e}")
        return df




In [ ]:

 for index, row in eligible_for_classification.iterrows():
    row_json_str = row.to_json()
    cc_project_json_record = json.loads(row_json_str)
    output = prompt_v4(cc_project_json_record, materials, cc_project_json_record['PRODUCT'], cc_project_json_record['PRODUCT_PRIORITY'])
            otpt = output.replace("```json", "").replace("```", "")  # Clean up any code block markers

    print(output)

```json
[
  {
    "project_id": 1006872527,
    "product": "Ceiling Sales",
    "product_priority": "high",
    "materials_matching": [
      "Acoustical Ceilings",
      "Drywall/Gypsum",
      "Metal Doors",
      "Firestopping",
      "Waterproofing",
      "Insulation",
      "Cold Formed Metal Framing",
      "Metal Decking",
      "Architectural Woodwork"
    ],
    "classification": "high",
    "reasoning": "The project involves addition/renovation of a fire/police station with a valuation of $4,637,163. The product priority is high.  The materials matching include Acoustical Ceilings, Drywall/Gypsum, Metal Doors, Firestopping, Waterproofing, Insulation, Cold Formed Metal Framing, Metal Decking and Architectural Woodwork.  Given the high product priority, a non-empty materials list, and the project valuation, the classification is 'high'."
  }
]
```
Error parsing XML: syntax error: line 1, column 0
```json
[
  {
    "project_id": 1006872527,
    "product": "Certainteed Drywall S

KeyboardInterrupt: 

In [291]:
output_df.to_csv('data/single_prompt_variations/prompt_classifier_role_output_60.csv')